# MS_B3 — Multi-Station GNN-LSTM + Stacking (Perfect Weather Forecast)

Trains GNN-LSTM + stacking ensemble for every station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast"` throughout (GNN node features and tabular
stacking base models both receive MET_COLS shifted to t+24).

**Checkpoint:** skips a station if `outputs/{station}/results/B3_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe
import src.models.gnn_stacking as gs

from src.config import (
    ALL_STATIONS, NEIGHBOR_STATIONS, HORIZONS, RANDOM_SEED,
    STATION_COORDS, get_station_paths,
)
from src.feature_engineering import build_feature_matrix, WEATHER_MODE_PERFECT
from src.models.gnn_stacking import (
    build_adjacency_matrix, build_gnn_sequence_dataset,
    train_gnn_lstm, predict_gnn_lstm,
    train_stacking_ensemble, predict_stacking,
)
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = WEATHER_MODE_PERFECT
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE}')

In [ ]:
# Stacking base model factories (same as B3 single-station notebook)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

def make_base_models():
    return {
        'xgb': lambda: TransformedTargetRegressor(
            regressor=XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                   random_state=RANDOM_SEED, verbosity=0, tree_method='hist'),
            func=np.log1p, inverse_func=np.expm1),
        'hgb': lambda: TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(
                max_iter=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_SEED),
            func=np.log1p, inverse_func=np.expm1),
        'lgb': lambda: TransformedTargetRegressor(
            regressor=lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                        random_state=RANDOM_SEED, verbose=-1),
            func=np.log1p, inverse_func=np.expm1),
        'cat': lambda: CatBoostRegressor(iterations=300, learning_rate=0.05, depth=4,
                                         random_seed=RANDOM_SEED, verbose=0, loss_function='MAE'),
    }

In [ ]:
SEQ_LEN_GNN = 24
META_FEAT_COLS_BASE = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'blh', 'pm25_now']

wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'B3_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')
    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so all src functions operate on the current station
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station
    gs.TARGET = station
    # ALL_STATIONS_ORDERED controls node order in GNN (target node must be index 0)
    gs.ALL_STATIONS_ORDERED = [station] + [s for s in NEIGHBOR_STATIONS if s != station]

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    # ── Build GNN sequence datasets ───────────────────────────────────
    X_gnn_train, y_gnn_train, A_static = build_gnn_sequence_dataset(
        train_df, seq_len=SEQ_LEN_GNN, horizons=HORIZONS, weather_mode=WEATHER_MODE
    )
    X_gnn_test, y_gnn_test, _ = build_gnn_sequence_dataset(
        test_df, seq_len=SEQ_LEN_GNN, horizons=HORIZONS, weather_mode=WEATHER_MODE
    )

    print(f'  GNN Train: {X_gnn_train.shape} | Test: {X_gnn_test.shape}')

    # 10% validation split from end of training sequences
    val_size = int(0.1 * len(X_gnn_train))
    X_gnn_tr  = X_gnn_train[:-val_size]
    y_gnn_tr  = y_gnn_train[:-val_size]
    X_gnn_val = X_gnn_train[-val_size:]
    y_gnn_val = y_gnn_train[-val_size:]

    # ── Train GNN-LSTM ────────────────────────────────────────────────
    gnn_model_path = paths['models'] / 'gnn_lstm_pfx_model.pt'
    gnn_model, _ = train_gnn_lstm(
        X_gnn_tr, y_gnn_tr, X_gnn_val, y_gnn_val, A_static,
        epochs=80, batch_size=64, patience=15, lr=1e-3,
        save_path=gnn_model_path
    )

    # GNN predictions (train for OOF alignment, test for final eval)
    gnn_train_preds = predict_gnn_lstm(gnn_model, X_gnn_train, A_static)  # (n_gnn_train, 24)
    gnn_test_preds  = predict_gnn_lstm(gnn_model, X_gnn_test,  A_static)  # (n_gnn_test, 24)

    # ── GNN evaluation ────────────────────────────────────────────────
    y_gnn_test_true = np.expm1(y_gnn_test)
    gnn_rows = []
    for h_idx, h in enumerate(HORIZONS):
        m = compute_metrics(y_gnn_test_true[:, h_idx], gnn_test_preds[:, h_idx])
        gnn_rows.append({'Model': 'B3_GNN_LSTM_pfx', 'Station': station, 'Horizon': h, **m})
    gnn_df = pd.DataFrame(gnn_rows)
    gnn_df.to_csv(paths['results'] / 'B3_GNN_metrics.csv', index=False)

    # ── Stacking ensemble ─────────────────────────────────────────────
    stack_models = {}
    stack_rows = []
    base_model_classes = make_base_models()

    for h in tqdm(HORIZONS, desc=f'Stacking [{station}]'):
        X_tr_h, y_tr_h = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        h_idx = h - 1

        # Align GNN OOF predictions length with tabular training rows
        n_gnn = len(gnn_train_preds)
        n_tab = len(X_tr_h)
        if n_gnn <= n_tab:
            gnn_oof_h = np.full(n_tab, np.nan)
            gnn_oof_h[-n_gnn:] = gnn_train_preds[:, h_idx]
            valid = ~np.isnan(gnn_oof_h)
            X_h_valid = X_tr_h[valid]
            y_h_valid = y_tr_h[valid]
            gnn_oof_valid = gnn_oof_h[valid]
        else:
            gnn_oof_valid = gnn_train_preds[-n_tab:, h_idx]
            X_h_valid = X_tr_h
            y_h_valid = y_tr_h

        meta, _ = train_stacking_ensemble(
            X_h_valid, y_h_valid,
            gnn_oof_preds=gnn_oof_valid,
            save_path=paths['models'] / f'meta_learner_pfx_h{h}.pkl'
        )
        stack_models[h] = meta

    # ── Stacking test evaluation ──────────────────────────────────────
    meta_feat_cols = None  # determined per-horizon below

    for h in HORIZONS:
        X_tr_h, y_tr_h = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        X_te_h, y_te_h = fe.build_feature_matrix(test_df,  horizon=h, weather_mode=WEATHER_MODE)

        if meta_feat_cols is None:
            meta_feat_cols = [c for c in META_FEAT_COLS_BASE if c in X_te_h.columns]

        base_preds_test = []
        for name, factory in base_model_classes.items():
            m = factory()
            m.fit(X_tr_h.values, y_tr_h.values)
            base_preds_test.append(m.predict(X_te_h.values))

        h_idx = h - 1
        n_gnn_te = len(gnn_test_preds)
        n_tab_te = len(X_te_h)
        gnn_h = np.zeros(n_tab_te)
        if n_gnn_te >= n_tab_te:
            gnn_h = gnn_test_preds[-n_tab_te:, h_idx]
        else:
            gnn_h[-n_gnn_te:] = gnn_test_preds[:, h_idx]

        base_preds_test.append(gnn_h)
        base_arr  = np.column_stack(base_preds_test)
        meta_feats = X_te_h[meta_feat_cols].values if meta_feat_cols else np.zeros((n_tab_te, 1))

        final_preds = predict_stacking(stack_models[h], base_arr, meta_feats)
        m = compute_metrics(y_te_h.values, final_preds)
        stack_rows.append({'Model': 'B3_Stacking_pfx', 'Station': station, 'Horizon': h, **m})

    stack_df = pd.DataFrame(stack_rows)
    stack_df.to_csv(paths['results'] / 'B3_Stacking_metrics.csv', index=False)

    # Combined B3 metrics (GNN + Stacking) — this file acts as the checkpoint
    combined = pd.concat([gnn_df, stack_df], ignore_index=True)
    combined.to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET and module state
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'
gs.TARGET = 'MzWarChrosci'
gs.ALL_STATIONS_ORDERED = ['MzWarChrosci'] + list(NEIGHBOR_STATIONS)

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')